# HGSplat — Weather-aware Heatmap Loss for LongSplat
**이윤호 / KNUVI**

| 섹션 | 내용 |
|------|------|
| 0 | GPU / CUDA 확인 |
| 1 | 레포 클론 및 의존성 설치 |
| 2 | Google Drive 마운트 및 경로 설정 |
| 3 | Heatmap 생성 (StyleFilter 인코더, 체크포인트 1개) |
| 4 | 학습 — Baseline |
| 5 | 학습 — Ours (Heatmap Loss) |
| 6 | 결과 비교 |

> **런타임:** 런타임 → 유형 변경 → **L4 GPU**

---
## Section 0 — GPU / CUDA 확인

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

---
## Section 1 — 레포 클론 및 의존성 설치

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_CODE = '/content/drive/MyDrive/HGSplat/code'

if os.path.isdir(DRIVE_CODE):
    print('[info] Drive에 레포 존재 → git pull')
    !git -C {DRIVE_CODE} pull
else:
    print('[info] 최초 설치 → git clone to Drive')
    !git clone --recursive https://github.com/Leeyoonho02/HGSplat.git {DRIVE_CODE}

# /content/HGSplat → Drive 경로 심볼릭 링크 (런타임마다)
!ln -sfn {DRIVE_CODE} /content/HGSplat
%cd /content/HGSplat

In [ ]:
import hashlib, os

REQ_FILE  = 'requirements.txt'
HASH_FILE = '/content/drive/MyDrive/HGSplat/.req_hash'

def file_hash(path):
    return hashlib.md5(open(path, 'rb').read()).hexdigest()

current_hash = file_hash(REQ_FILE)
stored_hash  = open(HASH_FILE).read().strip() if os.path.exists(HASH_FILE) else ''

if current_hash == stored_hash:
    print('[skip] requirements.txt 변경 없음 — pip install 스킵')
else:
    print('[info] requirements.txt 변경 감지 → pip install 실행')
    !pip install -r requirements.txt
    open(HASH_FILE, 'w').write(current_hash)
    print('[done] 설치 완료 및 해시 저장')

In [ ]:
import os
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

!pip install submodules/simple-knn --no-build-isolation
!pip install submodules/diff-gaussian-rasterization --no-build-isolation
!pip install submodules/fused-ssim --no-build-isolation

In [ ]:
# (선택) RoPE cuda 커널 컴파일 — 약 2분
!cd submodules/mast3r/dust3r/croco/models/curope && python setup.py build_ext --inplace
%cd /content/HGSplat

---
## Section 2 — 경로 설정

Drive 구조:
```
MyDrive/HGSplat/
├── code/                    ← 레포 (Section 1에서 자동 clone/pull)
├── scenes/
│   └── snow_scene/
│       ├── images/          ← 입력 프레임
│       └── heatmaps/        ← Section 3 실행 후 생성
└── checkpoints/
    └── style_filter.pth     ← MWFormer StyleFilter 체크포인트
```

In [ ]:
# ★ 여기만 수정
DRIVE_ROOT = '/content/drive/MyDrive/HGSplat'
SCENE_NAME = 'snow_scene'
SCENE_DIR  = f'{DRIVE_ROOT}/scenes/{SCENE_NAME}'
CKPT_STYLE = f'{DRIVE_ROOT}/checkpoints/style_filter.pth'

!mkdir -p data
!ln -sfn {SCENE_DIR} data/{SCENE_NAME}
!ls data/{SCENE_NAME}/

---
## Section 3 — Heatmap 생성

> Drive에 `heatmaps/` 폴더가 이미 있으면 스킵.

StyleFilter 인코더의 multi-scale feature map (채널 L2 norm → 업샘플 → 평균)으로 픽셀별 날씨 강도를 추정.  
복원 backbone(Network_top) 없이 **체크포인트 1개**만 사용.

In [ ]:
import os
heatmap_dir = f'data/{SCENE_NAME}/heatmaps'

if os.path.isdir(heatmap_dir) and len(os.listdir(heatmap_dir)) > 0:
    print(f'[skip] {heatmap_dir} 이미 존재 ({len(os.listdir(heatmap_dir))}개)')
else:
    print('[info] heatmap 생성 필요 — 아래 셀 실행')

In [ ]:
!python generate_heatmaps.py \
    --ckpt_style {CKPT_STYLE} \
    --scene_dir  data/{SCENE_NAME}/images \
    --out_dir    data/{SCENE_NAME}/heatmaps \
    --alpha      5.0

print('생성된 heatmap 수:', len(os.listdir(f'data/{SCENE_NAME}/heatmaps')))

---
## Section 4 — 학습: Baseline

`heatmaps/` 폴더가 없으면 자동으로 일반 L1 (Baseline)

In [ ]:
BASELINE_OUT = 'output/baseline'

# heatmaps/ 임시로 숨겨서 Baseline 강제
!mv data/{SCENE_NAME}/heatmaps data/{SCENE_NAME}/heatmaps_bak 2>/dev/null; true

!python train.py -s data/{SCENE_NAME} -m {BASELINE_OUT} --mode custom --resolution 2

!mv data/{SCENE_NAME}/heatmaps_bak data/{SCENE_NAME}/heatmaps 2>/dev/null; true

!python render.py  -m {BASELINE_OUT}
!python metrics.py -m {BASELINE_OUT}

---
## Section 5 — 학습: Ours (Heatmap Loss)

`heatmaps/` 폴더 존재 시 자동 활성화

In [ ]:
ALPHA    = 5.0
OURS_OUT = f'output/ours_alpha{ALPHA}'

!python train.py \
    -s data/{SCENE_NAME} \
    -m {OURS_OUT} \
    --mode custom \
    --resolution 2 \
    --heatmap_alpha {ALPHA}

!python render.py  -m {OURS_OUT}
!python metrics.py -m {OURS_OUT}

---
## Section 6 — 결과 비교

In [ ]:
import json, glob
import pandas as pd

def load_metrics(output_dir):
    paths = glob.glob(f'{output_dir}/**/results.json', recursive=True)
    if not paths:
        return None
    with open(paths[0]) as f:
        data = json.load(f)
    return list(data.values())[-1]

rows = []
for name, out in [('Baseline', BASELINE_OUT), (f'Ours (α={ALPHA})', OURS_OUT)]:
    m = load_metrics(out)
    rows.append({'Model': name, **(m if m else {'PSNR': '-', 'SSIM': '-', 'LPIPS': '-'})})
print(pd.DataFrame(rows).set_index('Model').to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def get_render_sample(d):
    imgs = sorted(glob.glob(f'{d}/**/renders/*.png', recursive=True))
    return imgs[0] if imgs else None

b_img = get_render_sample(BASELINE_OUT)
o_img = get_render_sample(OURS_OUT)

if b_img and o_img:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mpimg.imread(b_img)); axes[0].set_title('Baseline');          axes[0].axis('off')
    axes[1].imshow(mpimg.imread(o_img)); axes[1].set_title(f'Ours (α={ALPHA})'); axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('comparison.png', dpi=150)
    plt.show()
else:
    print('렌더링 결과 없음 — 학습 먼저 실행하세요.')

In [ ]:
import shutil, os
save_root = f'{DRIVE_ROOT}/results/{SCENE_NAME}'
os.makedirs(save_root, exist_ok=True)

for name, out in [('baseline', BASELINE_OUT), (f'ours_alpha{ALPHA}', OURS_OUT)]:
    if os.path.exists(out):
        shutil.copytree(out, f'{save_root}/{name}', dirs_exist_ok=True)
        print(f'[saved] {save_root}/{name}')

if os.path.exists('comparison.png'):
    shutil.copy('comparison.png', f'{save_root}/comparison.png')
    print(f'[saved] {save_root}/comparison.png')